In [1]:
from pathlib import Path
import sys
import os

# test_dictionary.ipynb is inside backend/test/
backend_dir = Path.cwd().parent

# Make sure this is the directory containing llm.py
assert (backend_dir / "llm.py").exists(), backend_dir

sys.path.insert(0, str(backend_dir))

import pandas as pd
from llm import detection
from dictionary_rewriting import dictionary_based_rewrite
import spacy
import random
import nltk
from transformers import AutoModelForTokenClassification, AutoTokenizer
nlp = spacy.load("it_core_news_lg")

c:\Users\fede9\OneDrive\Desktop\Fairly\fairly\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("set_n_gente.csv")

In [3]:
HUGGINFACE_TOKEN = os.getenv("HUGGINGFACE_TOKEN")
DETECTION_MODEL = "fairly-org/fairly_detection_v0"
TOKENIZER_MODEL = "osiria/roberta-base-italian"


In [4]:
sent_tokenizer = nltk.tokenize.punkt.PunktSentenceTokenizer()
detection_model = AutoModelForTokenClassification.from_pretrained(DETECTION_MODEL, token = HUGGINFACE_TOKEN)
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL, token = HUGGINFACE_TOKEN)

In [5]:
df.head()

,Unnamed: 0,SRC,REF-G,REF-N
0,750,They have shown that European issues can be wo...,Essi hanno dimostrato come sia possibile lavor...,Hanno dimostrato come sia possibile lavorare a...
1,751,As the European institutions acquire ever more...,Man mano che le Istituzioni europee ottengono ...,Man mano che le Istituzioni europee ottengono ...
2,752,The proposed Charter is unique in the way it g...,Nella Carta che viene proposta i cittadini e i...,Nella Carta che viene proposta la cittadinanza...
3,753,Discrimination between European and third coun...,La discriminazione fra europei e residenti dei...,La discriminazione fra cittadinanza europea e ...
4,754,"In addition, there are political gaps in the C...","Inoltre, la Carta presenta delle lacune politi...","Inoltre, la Carta presenta delle lacune politi..."


In [6]:
# Add colums for detection and rewriting
df["detection"] = None
df["rewriting_cv_0"] = None
df["rewriting_cv_1"] = None
df["rewriting_io"] = None
df["rewriting_iv"] = None

In [7]:
df_test = df[:5]
df_test.head()

,Unnamed: 0,SRC,REF-G,REF-N,detection,rewriting_cv_0,rewriting_cv_1,rewriting_io,rewriting_iv
0,750,They have shown that European issues can be wo...,Essi hanno dimostrato come sia possibile lavor...,Hanno dimostrato come sia possibile lavorare a...,None,None,None,None,None
1,751,As the European institutions acquire ever more...,Man mano che le Istituzioni europee ottengono ...,Man mano che le Istituzioni europee ottengono ...,None,None,None,None,None
2,752,The proposed Charter is unique in the way it g...,Nella Carta che viene proposta i cittadini e i...,Nella Carta che viene proposta la cittadinanza...,None,None,None,None,None
3,753,Discrimination between European and third coun...,La discriminazione fra europei e residenti dei...,La discriminazione fra cittadinanza europea e ...,None,None,None,None,None
4,754,"In addition, there are political gaps in the C...","Inoltre, la Carta presenta delle lacune politi...","Inoltre, la Carta presenta delle lacune politi...",None,None,None,None,None


In [8]:
STRATEGIES = ["CV-0", "CV-1", "IO-0", "IV-0"]

In [9]:
def transform_sentence(nlp, spans, sentence, strategy, gender_preference):
    output = []
    curr_idx = 0

    for span in sorted(spans, key=lambda s: s.start_char):
        start = span.start_char
        end = span.end_char
        # Avoid processing overlapping spans twice.
        if start < curr_idx:
            continue
        # Preserve text outside the detected span.
        output.append(sentence[curr_idx:start])
        rewritten_span = dictionary_based_rewrite(nlp, sentence, span, strategy, gender_preference)
        if rewritten_span is None:
            return "Almeno una parola di questa frase non è stata trovata nel dizionario!"
        else:
            output.append(span.reformulation)
        curr_idx = end
    
    # Preserve the remainder after the last span.
    output.append(sentence[curr_idx:])
    return "".join(output)

In [10]:
for i, row in df.iterrows():
    sentence = row["REF-G"]
    gender_preference = None
    # Perform detection
    spans = detection(sentence, sent_tokenizer, tokenizer, detection_model)
    rewritings = []
    output_detection = None
    if not spans:
        rewritings = [None, None, None, None]
    else:
        output_detection = " | ".join( [span.original_text for span in spans])
        for strategy in STRATEGIES:
            if strategy.split("-")[0] in ["CV", "IV"]:
                gender_preference = random.sample(["masculine", "feminine"], 1)[0]
            rewriting = transform_sentence(nlp, spans, sentence, strategy, gender_preference)
            rewritings.append(rewriting)
            # for span in spans:
            #     rewriting = dictionary_based_rewrite(nlp, sentence, span, strategy, gender_preference)
            #     rewritings.append(rewriting)

    df.loc[i, "detection"] = output_detection
    df.loc[i, "rewriting_cv_0"] = rewritings[0]
    df.loc[i, "rewriting_cv_1"] = rewritings[1]
    df.loc[i, "rewriting_io"] = rewritings[2]
    df.loc[i, "rewriting_iv"] = rewritings[3]
    

Span does not start with B-UNFAIR: residenti
Span does not start with B-UNFAIR: ,
Span does not start with B-UNFAIR: or
Span does not start with B-UNFAIR: ono
Span does not start with B-UNFAIR: ,
Span does not start with B-UNFAIR: cittadini
Span does not start with B-UNFAIR: nie
Span does not start with B-UNFAIR: li
Word vertice not found in the dictionary!
Word vertice not found in the dictionary!
Word vertice not found in the dictionary!
Word vertice not found in the dictionary!
Span does not start with B-UNFAIR: ed
Span does not start with B-UNFAIR: ,
Span does not start with B-UNFAIR: seguono
Word montaggio not found in the dictionary!
Word montaggio not found in the dictionary!
Word montaggio not found in the dictionary!
Word montaggio not found in the dictionary!
Span does not start with B-UNFAIR: soli
Span does not start with B-UNFAIR: Bundes
Span does not start with B-UNFAIR: Colonia
Word servizio not found in the dictionary!
Word servizio not found in the dictionary!
Word serv

In [11]:
df.head(10)

,Unnamed: 0,SRC,REF-G,REF-N,detection,rewriting_cv_0,rewriting_cv_1,rewriting_io,rewriting_iv
0,750,They have shown that European issues can be wo...,Essi hanno dimostrato come sia possibile lavor...,Hanno dimostrato come sia possibile lavorare a...,Essi hanno dimostrato | dei cittadini.,Esse e essi hanno dimostrata o dimostrato come...,Esse/Essi hanno dimostrata/dimostrato come sia...,Ess* hanno dimostrat* come sia possibile lavor...,Essi/Esse/Ess* hanno dimostrato/dimostrata/dim...
1,751,As the European institutions acquire ever more...,Man mano che le Istituzioni europee ottengono ...,Man mano che le Istituzioni europee ottengono ...,"sui cittadini,",Man mano che le Istituzioni europee ottengono ...,Man mano che le Istituzioni europee ottengono ...,Man mano che le Istituzioni europee ottengono ...,Man mano che le Istituzioni europee ottengono ...
2,752,The proposed Charter is unique in the way it g...,Nella Carta che viene proposta i cittadini e i...,Nella Carta che viene proposta la cittadinanza...,i cittadini,Nella Carta che viene proposta le cittadine e ...,Nella Carta che viene proposta i/le cittadini/...,Nella Carta che viene proposta l* cittadin* e ...,Nella Carta che viene proposta le/i/l* cittadi...
3,753,Discrimination between European and third coun...,La discriminazione fra europei e residenti dei...,La discriminazione fra cittadinanza europea e ...,europei | residenti,La discriminazione fra europei e europee e res...,La discriminazione fra europei/europee e resid...,La discriminazione fra europe* e residenti dei...,La discriminazione fra europee/europei/europe*...
4,754,"In addition, there are political gaps in the C...","Inoltre, la Carta presenta delle lacune politi...","Inoltre, la Carta presenta delle lacune politi...",None,None,None,None,None
5,755,"I am sad, indeed, to see the ranks of British ...",Pertanto mi rammarica vedere che i Conservator...,Pertanto mi rammarica vedere che il partito Co...,dei cittadini,Pertanto mi rammarica vedere che i Conservator...,Pertanto mi rammarica vedere che i Conservator...,Pertanto mi rammarica vedere che i Conservator...,Pertanto mi rammarica vedere che i Conservator...
6,756,Let us not merely adopt these rights but make ...,Non limitiamoci semplicemente ad approvare tal...,Non limitiamoci semplicemente ad approvare tal...,dei nostri cittadini:,Non limitiamoci semplicemente ad approvare tal...,Non limitiamoci semplicemente ad approvare tal...,Non limitiamoci semplicemente ad approvare tal...,Non limitiamoci semplicemente ad approvare tal...
7,757,The citizens of Europe have continued to be in...,I cittadini europei sono rimasti insufficiente...,La cittadinanza europea è rimasta insufficient...,I cittadini europei sono rimasti insufficiente...,I cittadini europei e le cittadine europee son...,Le/I cittadine/cittadini europee/europei sono ...,L* cittadin* europe* sono rimast* insufficient...,I/Le/L* cittadini/cittadine/cittadin* europei/...
8,758,Just one last word to the President-in-Office:...,Ancora un accenno al signor Presidente del Con...,Ancora un accenno alla Presidenza del Consigli...,None,None,None,None,None
9,759,"It is true, as one of the speakers said, that ...","È vero, come ha detto qualche oratore, che ser...","È vero, come ha detto qualche persona che ha p...","oratore,","È vero, come ha detto qualche oratrice o orato...","È vero, come ha detto qualche oratrice/oratore...","È vero, come ha detto qualche orator* , che se...","È vero, come ha detto qualche oratrice/oratore..."


In [17]:
df_final = df[["SRC", "REF-G", "detection", "rewriting_cv_0", "rewriting_cv_1", "rewriting_io", "rewriting_iv"]]

In [19]:
len(df_final)

750

In [20]:
df_final.to_csv("gente_test_rewritings.csv", encoding="utf-8")

In [12]:
count_missing = []
for _, row in df.iterrows():
    rew = row["rewriting_cv_0"]
    if rew == "Almeno una parola di questa frase non è stata trovata nel dizionario!":
        count_missing.append(row["REF-G"])

In [13]:
len(count_missing)

11

In [15]:
df_missing = df[df["rewriting_cv_0"] == "Almeno una parola di questa frase non è stata trovata nel dizionario!"]

In [16]:
df_missing.to_csv("missing_words.csv", encoding="utf-8")

In [17]:
sent = "Oltre ad adottare tre importanti documenti, i leader che hanno preso parte al vertice sono stati impegnati attivamente nel discutere questioni delicate quali i diritti dell'uomo, il mare della Cina meridionale, Timor Est e la Birmania."
doc = nlp(sent)
for token in doc:
    print(f"{token.text} -> {token.pos_}")
    print( token.morph.to_dict())

Oltre -> ADV
{}
ad -> ADP
{}
adottare -> VERB
{'VerbForm': 'Inf'}
tre -> NUM
{'NumType': 'Card'}
importanti -> ADJ
{'Number': 'Plur'}
documenti -> NOUN
{'Gender': 'Masc', 'Number': 'Plur'}
, -> PUNCT
{}
i -> DET
{'Definite': 'Def', 'Gender': 'Masc', 'Number': 'Plur', 'PronType': 'Art'}
leader -> NOUN
{}
che -> PRON
{'PronType': 'Rel'}
hanno -> AUX
{'Mood': 'Ind', 'Number': 'Plur', 'Person': '3', 'Tense': 'Pres', 'VerbForm': 'Fin'}
preso -> VERB
{'Gender': 'Masc', 'Number': 'Sing', 'Tense': 'Past', 'VerbForm': 'Part'}
parte -> NOUN
{'Gender': 'Fem', 'Number': 'Sing'}
al -> ADP
{'Definite': 'Def', 'Gender': 'Masc', 'Number': 'Sing', 'PronType': 'Art'}
vertice -> NOUN
{'Gender': 'Masc', 'Number': 'Sing'}
sono -> AUX
{'Mood': 'Ind', 'Number': 'Plur', 'Person': '3', 'Tense': 'Pres', 'VerbForm': 'Fin'}
stati -> AUX
{'Gender': 'Masc', 'Number': 'Plur', 'Tense': 'Past', 'VerbForm': 'Part'}
impegnati -> VERB
{'Gender': 'Masc', 'Number': 'Plur', 'Tense': 'Past', 'VerbForm': 'Part'}
attivamente -

In [ ]:
df.drop